# Layer V ET Identity Gene Analysis
Identifies genes that define Layer V ET identity through stepwise comparison:
- **Non-Neuron vs Layer V ET** → Neuron-identity genes
- **Upper Layer vs Layer V ET** → Deep-layer-identity genes  
- **Other Deep Layer vs Layer V ET** → Layer V ET-specific genes

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

sc.settings.verbosity = 1

/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: 

## 1. Load Data

In [4]:
# ── EDIT THIS PATH ──────────────────────────────────────────────────────────
H5AD_PATH = "/home/nakagawa/datasets/h5ad/SMARTer_cells_MOp.h5ad"
OUT_DIR   = Path("/home/nakagawa/datasets/LayerV_ET_results")
# ────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
adata = sc.read_h5ad(H5AD_PATH)
print(adata)

AnnData object with n_obs × n_vars = 6288 × 35211
    obs: 'BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'ar_id', 'exp_component_vendor_name', 'batch', 'batch_vendor_name', 'tube', 'tube_internal_name', 'tube_contents_nm', 'tube_contents_nm_from_vendor', 'tube_avg_size_bp', 'tube_input_fmol', 'r1_index', 'r2_index', 'index_sequence_pair', 'facs_container', 'sample_name', 'patched_cell_container', 'cell_name', 'cell_id', 'sample_quantity_count', 'sample_quantity_pg', 'donor_id', 'control', 'full_genotype', 'facs_population_plan', 'cre_line', 'reporter', 'injection_roi', 'injection_materials', 'roi', 'patchseq_roi', 'medical_conditions', 'slice_min_pos', 'slice_max_pos', 'rna_amplification_set', 'rna_amplification', 'amp_date', 'pcr_cycles', 'percent_cdna_longer_than_400bp', 'rna_amplification_pass_fail', 'amplified_quantity_ng', 'library_prep_set', 'library_prep', 'lib_date', 'library_input_ng', 'avg_size_bp', 'quantification2_ng',

## 2. Find the Cell Type Column & List All Cell Types

In [8]:
# Show all metadata columns so you can identify the right cell type column
print("Available metadata columns:")
print(adata.obs.columns.tolist())
print(adata.obs['BICCN_subclass_label'].value_counts())

Available metadata columns:
['BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'ar_id', 'exp_component_vendor_name', 'batch', 'batch_vendor_name', 'tube', 'tube_internal_name', 'tube_contents_nm', 'tube_contents_nm_from_vendor', 'tube_avg_size_bp', 'tube_input_fmol', 'r1_index', 'r2_index', 'index_sequence_pair', 'facs_container', 'sample_name', 'patched_cell_container', 'cell_name', 'cell_id', 'sample_quantity_count', 'sample_quantity_pg', 'donor_id', 'control', 'full_genotype', 'facs_population_plan', 'cre_line', 'reporter', 'injection_roi', 'injection_materials', 'roi', 'patchseq_roi', 'medical_conditions', 'slice_min_pos', 'slice_max_pos', 'rna_amplification_set', 'rna_amplification', 'amp_date', 'pcr_cycles', 'percent_cdna_longer_than_400bp', 'rna_amplification_pass_fail', 'amplified_quantity_ng', 'library_prep_set', 'library_prep', 'lib_date', 'library_input_ng', 'avg_size_bp', 'quantification2_ng', 'quantification_fmol', 'quant

In [9]:
# ── EDIT: set to the column name that contains cell type labels ──────────────
# Common names: 'cell_type', 'CellType', 'cluster', 'subclass_label', 'cell_type_alias_label'
CELLTYPE_COL = "BICCN_subclass_label"   # <-- change this if needed after checking output above
# ────────────────────────────────────────────────────────────────────────────

cell_types = sorted(adata.obs[CELLTYPE_COL].unique().tolist())
print(f"\nFound {len(cell_types)} cell types in '{CELLTYPE_COL}':\n")
for i, ct in enumerate(cell_types):
    n = (adata.obs[CELLTYPE_COL] == ct).sum()
    print(f"  [{i:02d}] {ct}  (n={n})")


Found 17 cell types in 'BICCN_subclass_label':

  [00] Astro  (n=10)
  [01] Endo  (n=7)
  [02] L2/3 IT  (n=483)
  [03] L5 ET  (n=12)
  [04] L5 IT  (n=1572)
  [05] L5/6 NP  (n=210)
  [06] L6 CT  (n=904)
  [07] L6 IT  (n=395)
  [08] L6 IT Car3  (n=5)
  [09] L6b  (n=571)
  [10] Lamp5  (n=377)
  [11] Pvalb  (n=543)
  [12] SMC  (n=21)
  [13] Sncg  (n=84)
  [14] Sst  (n=429)
  [15] VLMC  (n=6)
  [16] Vip  (n=659)


## 3. Define Layer V ET and Classify Other Cell Types
After seeing the list above, fill in the three categories below.

In [21]:
# ── EDIT THESE after reading cell type list above ────────────────────────────

LAYER_V_ET = "L5 ET"   # exact string from the list above

# Copy-paste cell type names from the printed list into each group
NON_NEURON_TYPES = [
    "Astro",       # Astrocyte
    "Endo",        # Endothelial
    "SMC",         # Smooth Muscle Cell
    "VLMC",        # Vascular Leptomeningeal Cell
]

UPPER_LAYER_TYPES = [
    "L2/3 IT",     # Classic upper layer
]

OTHER_DEEP_LAYER_TYPES = [
    "L5 IT",       # Deep but NOT ET
    "L5/6 NP",     # Near-projecting, deep layer
    "L6 CT",       # Corticothalamic
    "L6 IT",       # Deep IT
    #"L6 IT Car3",  # Deep IT subtype but only has n=6 
    "L6b",         # Subplate-like deep layer
]

# Your target
TARGET_TYPE = "L5 ET"

# Interneurons — consider excluding or treating separately
INTERNEURON_TYPES = [
    "Lamp5",
    "Pvalb",
    "Sncg",
    "Sst",
    "Vip",
]

# ── Thresholds ───────────────────────────────────────────────────────────────
LOG2FC_THRESH  = 1.0    # absolute log2 fold change cutoff (2-fold)
PVAL_THRESH    = 0.05   # adjusted p-value (FDR)
# ─────────────────────────────────────────────────────────────────────────────

print(f"Target: {LAYER_V_ET}")
print(f"Non-neuron types  : {NON_NEURON_TYPES}")
print(f"Upper layer types : {UPPER_LAYER_TYPES}")
print(f"Other deep layer  : {OTHER_DEEP_LAYER_TYPES}")
print(f"GABAergic inhibitory neurons : {INTERNEURON_TYPES}")

Target: L5 ET
Non-neuron types  : ['Astro', 'Endo', 'SMC', 'VLMC']
Upper layer types : ['L2/3 IT']
Other deep layer  : ['L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6b']
GABAergic inhibitory neurons : ['Lamp5', 'Pvalb', 'Sncg', 'Sst', 'Vip']


## 4. Preprocessing

In [23]:
# Use raw counts if available, otherwise use .X
if adata.raw is not None:
    print("Using adata.raw for DE analysis")
    adata_use = adata.raw.to_adata()
else:
    print("Using adata.X for DE analysis")
    adata_use = adata.copy()

# Copy cell type labels to adata_use
adata_use.obs[CELLTYPE_COL] = adata.obs[CELLTYPE_COL]

# Normalize if not already (check if values look like raw counts)
if adata_use.X.max() > 100:
    sc.pp.normalize_total(adata_use, target_sum=1e4)
    sc.pp.log1p(adata_use)
    print("Normalized and log1p transformed")
else:
    print("Data appears pre-normalized")

Using adata.raw for DE analysis
Normalized and log1p transformed


## 5. Run DE Analysis: Each Cell Type vs Layer V ET

In [24]:
def run_de(adata_use, celltype_col, group_a, group_b, log2fc_thresh, pval_thresh):
    """
    Run Wilcoxon rank-sum DE between group_a vs group_b.
    Positive log2FC = upregulated in group_a relative to group_b (Layer V ET).
    """
    mask = adata_use.obs[celltype_col].isin([group_a, group_b])
    sub  = adata_use[mask].copy()
    sub.obs["group"] = sub.obs[celltype_col].astype(str)

    sc.tl.rank_genes_groups(
        sub,
        groupby="group",
        groups=[group_a],
        reference=group_b,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
    )

    result = sc.get.rank_genes_groups_df(sub, group=group_a)
    result = result.rename(columns={
        "names"       : "gene",
        "logfoldchanges": "log2FC",
        "pvals_adj"   : "padj",
        "pvals"       : "pval",
        "scores"      : "score",
    })

    result["-log10padj"] = -np.log10(result["padj"].clip(lower=1e-300))
    result["comparison"] = f"{group_a}_vs_LayerVET"

    sig = result[
        (result["padj"] < pval_thresh) &
        (result["log2FC"].abs() >= log2fc_thresh)
    ].copy()

    sig["direction"] = np.where(sig["log2FC"] > 0, "UP_in_other", "UP_in_LayerVET")

    return result, sig


all_comparisons   = {}   # full results
all_sig           = {}   # significant only

all_cell_types = NON_NEURON_TYPES + UPPER_LAYER_TYPES + OTHER_DEEP_LAYER_TYPES + INTERNEURON_TYPES

for ct in all_cell_types:
    if ct not in adata_use.obs[CELLTYPE_COL].values:
        print(f"[SKIP] '{ct}' not found in data")
        continue
    print(f"[DE]  {ct} vs {LAYER_V_ET} ...", end=" ")
    full, sig = run_de(adata_use, CELLTYPE_COL, ct, LAYER_V_ET, LOG2FC_THRESH, PVAL_THRESH)
    all_comparisons[ct] = full
    all_sig[ct]         = sig
    print(f"{len(sig)} significant genes (UP_other={( sig.direction=='UP_in_other').sum()}, UP_LayerVET={(sig.direction=='UP_in_LayerVET').sum()})")

[DE]  Astro vs L5 ET ... 3969 significant genes (UP_other=44, UP_LayerVET=3925)
[DE]  Endo vs L5 ET ... 3981 significant genes (UP_other=259, UP_LayerVET=3722)
[DE]  SMC vs L5 ET ... 6056 significant genes (UP_other=262, UP_LayerVET=5794)
[DE]  VLMC vs L5 ET ... 3922 significant genes (UP_other=163, UP_LayerVET=3759)
[DE]  L2/3 IT vs L5 ET ... 1169 significant genes (UP_other=520, UP_LayerVET=649)
[DE]  L5 IT vs L5 ET ... 876 significant genes (UP_other=311, UP_LayerVET=565)
[DE]  L5/6 NP vs L5 ET ... 1245 significant genes (UP_other=312, UP_LayerVET=933)
[DE]  L6 CT vs L5 ET ... 1013 significant genes (UP_other=264, UP_LayerVET=749)
[DE]  L6 IT vs L5 ET ... 1037 significant genes (UP_other=378, UP_LayerVET=659)
[DE]  L6b vs L5 ET ... 1295 significant genes (UP_other=375, UP_LayerVET=920)
[DE]  Lamp5 vs L5 ET ... 1743 significant genes (UP_other=361, UP_LayerVET=1382)
[DE]  Pvalb vs L5 ET ... 1877 significant genes (UP_other=439, UP_LayerVET=1438)
[DE]  Sncg vs L5 ET ... 2039 significa

## 6. Save Individual Comparison CSVs

In [25]:
for ct, sig in all_sig.items():
    safe_name = ct.replace("/", "_").replace(" ", "_")
    path = OUT_DIR / f"{safe_name}_vs_LayerVET.csv"
    sig.sort_values("log2FC", ascending=False).to_csv(path, index=False)
    print(f"Saved: {path.name}")

Saved: Astro_vs_LayerVET.csv
Saved: Endo_vs_LayerVET.csv
Saved: SMC_vs_LayerVET.csv
Saved: VLMC_vs_LayerVET.csv
Saved: L2_3_IT_vs_LayerVET.csv
Saved: L5_IT_vs_LayerVET.csv
Saved: L5_6_NP_vs_LayerVET.csv
Saved: L6_CT_vs_LayerVET.csv
Saved: L6_IT_vs_LayerVET.csv
Saved: L6b_vs_LayerVET.csv
Saved: Lamp5_vs_LayerVET.csv
Saved: Pvalb_vs_LayerVET.csv
Saved: Sncg_vs_LayerVET.csv
Saved: Sst_vs_LayerVET.csv
Saved: Vip_vs_LayerVET.csv


## 7. Stepwise Candidate Narrowing

Logic:
- **Neuron-identity genes** = UP in Layer V ET vs ANY non-neuron type
- **Exitory-Neuron-identity genes** = above AND UP in Layer V ET vs ANY interneurons type
- **Deep-layer-identity genes** = above AND UP in Layer V ET vs ANY upper layer type
- **Layer V ET-specific genes** = above AND UP in Layer V ET vs ALL other deep layer types

In [26]:
def get_layerVET_up_genes(all_sig, cell_types):
    """Genes consistently UP in Layer V ET (direction == UP_in_LayerVET) across given comparisons."""
    sets = []
    for ct in cell_types:
        if ct in all_sig:
            up = set(all_sig[ct][all_sig[ct]["direction"] == "UP_in_LayerVET"]["gene"])
            sets.append(up)
    if not sets:
        return set()
    # Gene must appear in AT LEAST ONE comparison to be included at each step
    return set.union(*sets)


# Step 1: Neuron identity — UP in Layer V ET vs non-neurons
neuron_identity_genes = get_layerVET_up_genes(all_sig, NON_NEURON_TYPES)
print(f"Step 1 — Neuron-identity genes (UP vs non-neurons):  {len(neuron_identity_genes)}")

# Step 2: Exitory Neuron identity — above AND UP vs interneuron types
interneuron_up = get_layerVET_up_genes(all_sig, INTERNEURON_TYPES)
exitory_neuron_identity_genes = neuron_identity_genes & interneuron_up
print(f"Step 2 — Exitory-Neuron-identity genes (also UP vs interneurons):  {len(exitory_neuron_identity_genes)}")

# Step 3: Deep layer identity — above AND UP vs upper layer types
upper_layer_up = get_layerVET_up_genes(all_sig, UPPER_LAYER_TYPES)
deep_layer_identity_genes = exitory_neuron_identity_genes & upper_layer_up
print(f"Step 3 — Deep-layer-identity genes (also UP vs upper layer): {len(deep_layer_identity_genes)}")

# Step 4: Layer V ET specific — above AND UP vs ALL other deep layer types
# Must be UP in Layer V ET in EVERY other-deep-layer comparison (intersection)
deep_layer_sets = []
for ct in OTHER_DEEP_LAYER_TYPES:
    if ct in all_sig:
        up = set(all_sig[ct][all_sig[ct]["direction"] == "UP_in_LayerVET"]["gene"])
        deep_layer_sets.append(up)

if deep_layer_sets:
    consistently_up_vs_deep = set.intersection(*deep_layer_sets)
    layerVET_specific_genes = deep_layer_identity_genes & consistently_up_vs_deep
else:
    layerVET_specific_genes = set()

print(f"Step 3 — Layer V ET-specific genes (UP vs ALL other deep layer): {len(layerVET_specific_genes)}")

Step 1 — Neuron-identity genes (UP vs non-neurons):  7713
Step 2 — Exitory-Neuron-identity genes (also UP vs interneurons):  2391
Step 3 — Deep-layer-identity genes (also UP vs upper layer): 452
Step 3 — Layer V ET-specific genes (UP vs ALL other deep layer): 108


## 8. Build Summary Tables & Save

In [31]:
# ── Gene symbol mapping ───────────────────────────────────────────────────────
id_to_symbol = adata.var['feature_name'].to_dict()
print(f"Loaded {len(id_to_symbol)} gene symbols — example: {list(id_to_symbol.items())[:3]}")

# ── Collect stats for each candidate gene across all comparisons ──────────────
def build_summary(gene_set, all_comparisons, label):
    rows = []
    for gene in sorted(gene_set):
        row = {
            "gene":        gene,
            "gene_symbol": id_to_symbol.get(gene, gene),  # fallback to ID if not found
            "category":    label,
        }
        for ct, df in all_comparisons.items():
            match = df[df["gene"] == gene]
            if not match.empty:
                row[f"{ct}__log2FC"]    = round(match["log2FC"].values[0], 3)
                row[f"{ct}__log10padj"] = round(match["-log10padj"].values[0], 3)
                row[f"{ct}__padj"]      = match["padj"].values[0]
        rows.append(row)
    return pd.DataFrame(rows)

df_neuron   = build_summary(neuron_identity_genes,          all_comparisons, "neuron_identity")
df_exitory  = build_summary(exitory_neuron_identity_genes,  all_comparisons, "exitory_neuron_identity")
df_deep     = build_summary(deep_layer_identity_genes,      all_comparisons, "deep_layer_identity")
df_specific = build_summary(layerVET_specific_genes,        all_comparisons, "LayerVET_specific")

# ── Save ──────────────────────────────────────────────────────────────────────
df_neuron.to_csv(  OUT_DIR / "neuron_identity_genes.csv",          index=False)
df_exitory.to_csv( OUT_DIR / "exitory_neuron_identity_genes.csv",  index=False)
df_deep.to_csv(    OUT_DIR / "deep_layer_identity_genes.csv",      index=False)
df_specific.to_csv(OUT_DIR / "LayerVET_specific_genes.csv",        index=False)

print(f"Saved to {OUT_DIR}/")
print(f"  neuron_identity_genes.csv              : {len(df_neuron)} genes")
print(f"  exitory_neuron_identity_genes.csv      : {len(df_exitory)} genes")
print(f"  deep_layer_identity_genes.csv          : {len(df_deep)} genes")
print(f"  LayerVET_specific_genes.csv            : {len(df_specific)} genes")

Loaded 35211 gene symbols — example: [('ENSMUSG00000086260', 'Gm16259'), ('ENSMUSG00000031429', 'Psmd10'), ('ENSMUSG00000094053', 'Scgb2b7')]
Saved to /home/nakagawa/datasets/LayerV_ET_results/
  neuron_identity_genes.csv              : 7713 genes
  exitory_neuron_identity_genes.csv      : 2391 genes
  deep_layer_identity_genes.csv          : 452 genes
  LayerVET_specific_genes.csv            : 108 genes


## 9. Preview Results

In [36]:
print("\n=== TOP 20 Layer V ET-SPECIFIC GENES ===")
if not df_specific.empty:
    log2fc_cols = [c for c in df_specific.columns if c.endswith("__log2FC")]
    df_specific["mean_log2FC"] = df_specific[log2fc_cols].mean(axis=1)
    display(df_specific[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]  # ← added gene_symbol
            .sort_values("mean_log2FC", ascending=False)
            .head(20)
            .reset_index(drop=True))
else:
    print("No Layer V ET-specific genes found — consider relaxing thresholds (LOG2FC_THRESH / PVAL_THRESH)")

print("\n=== TOP 20 DEEP LAYER IDENTITY GENES ===")
if not df_deep.empty:
    log2fc_cols = [c for c in df_deep.columns if c.endswith("__log2FC")]
    df_deep["mean_log2FC"] = df_deep[log2fc_cols].mean(axis=1)
    display(df_deep[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]  # ← added gene_symbol
            .sort_values("mean_log2FC", ascending=False)
            .head(20)
            .reset_index(drop=True))


=== TOP 20 Layer V ET-SPECIFIC GENES ===


,gene,gene_symbol,mean_log2FC,Astro__log2FC,Endo__log2FC,SMC__log2FC,VLMC__log2FC,L2/3 IT__log2FC,L5 IT__log2FC,L5/6 NP__log2FC,L6 CT__log2FC,L6 IT__log2FC,L6b__log2FC,Lamp5__log2FC,Pvalb__log2FC,Sncg__log2FC,Sst__log2FC,Vip__log2FC
0,ENSMUSG00000031765,Mt1,-0.735200,3.372000,2.277,-1.874000,-0.013000,-1.216,-2.567,-1.534,-1.527,-1.137,-1.727,-0.915,-0.097,-1.011,-2.231,-0.828
1,ENSMUSG00000025732,Mcrip2,-1.213800,-0.825000,-0.163,-1.754000,-1.182000,-2.154,-2.208,-1.097,-1.925,-1.997,-1.395,-0.505,0.419,-1.442,-1.005,-0.974
2,ENSMUSG00000017561,Crlf3,-1.240867,1.137000,-3.840,0.715000,-0.320000,-2.169,-2.054,-1.569,-1.143,-1.536,-1.235,-1.773,-0.433,-1.400,-1.582,-1.411
3,ENSMUSG00000001089,Luzp1,-1.330467,-4.504000,0.763,0.602000,-0.861000,-1.091,-1.061,-2.385,-2.109,-1.648,-1.859,-1.286,-0.046,-1.520,-1.493,-1.459
4,ENSMUSG00000027712,Anxa5,-1.414133,-2.212000,2.839,1.921000,1.693000,-1.844,-1.492,-1.457,-2.326,-2.179,-4.657,-2.790,-1.826,-1.725,-2.599,-2.558
5,ENSMUSG00000026456,Cyb5r1,-1.567667,-4.410000,0.916,-0.766000,-1.027000,-2.621,-3.046,-1.049,-2.210,-1.547,-1.308,-0.917,-1.267,-0.944,-1.497,-1.822
6,ENSMUSG00000020823,Sec14l1,-1.914067,-3.237000,0.215,-2.713000,-2.668000,-1.426,-1.412,-2.068,-2.089,-2.078,-3.682,-1.196,-1.648,-0.825,-2.752,-1.132
7,ENSMUSG00000018334,Ksr1,-1.914133,-2.356000,0.168,-2.190000,0.311000,-3.361,-2.587,-1.848,-1.672,-2.062,-2.118,-1.923,-1.425,-2.179,-3.509,-1.961
8,ENSMUSG00000061046,Haghl,-1.941000,-1.890000,-3.832,-3.328000,-3.203000,-2.031,-1.979,-1.136,-2.114,-2.094,-1.845,-1.200,-0.453,-1.405,-0.980,-1.625
9,ENSMUSG00000030970,Ctbp2,-1.948267,-3.794000,0.679,0.182000,1.151000,-2.111,-1.817,-1.410,-4.675,-5.197,-3.137,-3.534,-0.690,-2.598,-1.531,-0.742



=== TOP 20 DEEP LAYER IDENTITY GENES ===


,gene,gene_symbol,mean_log2FC,Astro__log2FC,Endo__log2FC,SMC__log2FC,VLMC__log2FC,L2/3 IT__log2FC,L5 IT__log2FC,L5/6 NP__log2FC,L6 CT__log2FC,L6 IT__log2FC,L6b__log2FC,Lamp5__log2FC,Pvalb__log2FC,Sncg__log2FC,Sst__log2FC,Vip__log2FC
0,ENSMUSG00000031604,Msmo1,-0.256533,3.108,-1.000,-1.765,-0.903,-1.468,-0.377,1.041,0.129,0.554,1.206,-1.071,-1.801,0.058,-0.816,-0.743
1,ENSMUSG00000079523,Tmsb10,-0.602933,-3.522,0.091,-1.185,1.116,-1.866,-0.498,0.839,1.037,-1.279,1.261,-1.569,-4.138,0.164,0.389,0.116
2,ENSMUSG00000021665,Hexb,-0.665533,-2.862,0.034,-2.039,2.909,-1.572,-0.748,-0.071,-0.356,-0.637,-0.280,-1.580,-0.154,-0.742,-0.659,-1.226
3,ENSMUSG00000018585,Atox1,-0.668200,-0.689,2.246,-1.225,-0.400,-1.005,-0.953,-0.718,-0.908,-0.980,-0.635,-1.037,-0.753,-1.111,-0.793,-1.062
4,ENSMUSG00000035642,Aamdc,-0.727667,-1.105,-0.651,-1.633,0.717,-1.213,-1.018,-0.624,-0.443,-0.813,-0.861,-0.440,-0.290,-0.857,-0.598,-1.086
5,ENSMUSG00000031765,Mt1,-0.735200,3.372,2.277,-1.874,-0.013,-1.216,-2.567,-1.534,-1.527,-1.137,-1.727,-0.915,-0.097,-1.011,-2.231,-0.828
6,ENSMUSG00000091955,Tmsb10b,-0.861200,-4.731,-0.112,-1.477,0.006,-1.821,-0.550,0.834,1.126,-1.263,1.292,-1.647,-4.591,-0.102,0.199,-0.081
7,ENSMUSG00000064254,Ethe1,-1.009067,-1.123,-0.316,-2.053,-1.087,-1.568,-0.926,0.009,-0.540,-1.091,-0.291,-1.671,0.033,-1.647,-0.903,-1.962
8,ENSMUSG00000019864,Rtn4ip1,-1.049267,0.149,0.738,-2.732,-1.801,-1.153,-1.190,-0.652,-1.691,-0.992,-1.471,-0.812,-1.054,-0.525,-1.284,-1.269
9,ENSMUSG00000078453,Abracl,-1.073733,-5.092,-1.006,-2.062,-1.669,-1.502,0.831,0.193,1.070,0.119,0.879,-1.605,-1.917,-1.326,-1.421,-1.598


## 10. Threshold Sensitivity Check (Optional)
Run this if Step 3 returns too few or too many genes.

In [34]:
print(adata.var.columns.tolist())
print(adata.var.head())

['feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
                    feature_is_filtered feature_name feature_reference  \
ENSMUSG00000086260                False      Gm16259   NCBITaxon:10090   
ENSMUSG00000031429                False       Psmd10   NCBITaxon:10090   
ENSMUSG00000094053                False      Scgb2b7   NCBITaxon:10090   
ENSMUSG00000116093                False       Gm3888   NCBITaxon:10090   
ENSMUSG00000048003                False     Catsper4   NCBITaxon:10090   

                   feature_biotype feature_length          feature_type  
ENSMUSG00000086260            gene            553  processed_pseudogene  
ENSMUSG00000031429            gene           1410        protein_coding  
ENSMUSG00000094053            gene            583        protein_coding  
ENSMUSG00000116093            gene           1240  processed_pseudogene  
ENSMUSG00000048003            gene           1069        protein_coding

In [35]:
print("Sensitivity check — gene counts at different thresholds:\n")
print(f"{'log2FC':>8}  {'padj':>6}  {'Neuron':>8}  {'DeepLayer':>10}  {'LayerVET_specific':>18}")

for lfc in [0.5, 1.0, 1.5, 2.0]:
    for pv in [0.1, 0.05, 0.01]:
        def get_up(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    up = set(df[(df["padj"] < pv) & (df["log2FC"] < -lfc)]["gene"])
                    sets.append(up)
            return set.union(*sets) if sets else set()

        def get_up_intersect(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    up = set(df[(df["padj"] < pv) & (df["log2FC"] < -lfc)]["gene"])
                    sets.append(up)
            return set.intersection(*sets) if sets else set()

        n  = len(get_up(NON_NEURON_TYPES))
        ul = len(get_up(NON_NEURON_TYPES) & get_up(UPPER_LAYER_TYPES))
        sp = len(get_up(NON_NEURON_TYPES) & get_up(UPPER_LAYER_TYPES) & get_up_intersect(OTHER_DEEP_LAYER_TYPES))
        print(f"{lfc:>8.1f}  {pv:>6.2f}  {n:>8}  {ul:>10}  {sp:>18}")

Sensitivity check — gene counts at different thresholds:

  log2FC    padj    Neuron   DeepLayer   LayerVET_specific
     0.5    0.10      8806        1166                 267
     0.5    0.05      7935         920                 200
     0.5    0.01      5589         525                 115
     1.0    0.10      8579         690                 137
     1.0    0.05      7713         570                 110
     1.0    0.01      5384         347                  66
     1.5    0.10      8213         402                  82
     1.5    0.05      7394         351                  70
     1.5    0.01      5081         240                  48
     2.0    0.10      7781         269                  41
     2.0    0.05      7052         231                  34
     2.0    0.01      4750         161                  24
